# Pandas Data Exploration & Cleaning + Delta Lake MERGE
**Assignment:** Python Basics | Data Exploration | Data Cleaning | Delta Lake MERGE Implementation 
**Dataset:** Superstore Sales Dataset 
**Reference:** [Delta Lake MERGE — Microsoft Learn](https://learn.microsoft.com/en-us/azure/databricks/delta/merge)

---


## 0. Setup & Imports

In [ ]:
# Install required libraries (run once)
# !pip install pandas numpy delta-spark pyspark

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print(" Libraries imported successfully")
print(f" Pandas version : {pd.__version__}")
print(f" NumPy version : {np.__version__}")


## Step 1 — Load CSV Dataset into a Pandas DataFrame

We load the Superstore Sales dataset using `pd.read_csv()`. 
This dataset contains transactional sales data with columns like Sales, Quantity, Profit, Region, Category, etc.


In [ ]:
# Load the dataset
df = pd.read_csv('superstore_raw.csv')

print(f" Dataset loaded successfully!")
print(f" Rows : {df.shape[0]}")
print(f" Columns : {df.shape[1]}")


## Step 2 — Data Exploration

Explore the dataset using standard Pandas methods:
- `head()` / `tail()` — view first / last rows
- `shape` — dimensions
- `columns` — column names
- `dtypes` — data types
- `describe()` — summary statistics
- `info()` — concise overview


In [ ]:
# 2a. First 5 rows
print("=== HEAD (first 5 rows) ===")
df.head()


In [ ]:
# 2b. Last 5 rows
print("=== TAIL (last 5 rows) ===")
df.tail()


In [ ]:
# 2c. Shape — (rows, columns)
print(f"Shape: {df.shape} → {df.shape[0]} rows × {df.shape[1]} columns")


In [ ]:
# 2d. Column names
print("Columns:")
for i, col in enumerate(df.columns, 1):
 print(f" {i:2d}. {col}")


In [ ]:
# 2e. Data types of each column
print("=== DATA TYPES ===")
print(df.dtypes)


In [ ]:
# 2f. Summary statistics for numeric columns
print("=== DESCRIPTIVE STATISTICS ===")
df.describe().round(2)


In [ ]:
# 2g. Concise info — non-null counts + dtypes
print("=== DATASET INFO ===")
df.info()


## Step 3 — Handle Missing Values

### 3a. Identify missing values
We check how many null values each column has.


In [ ]:
# Count missing values per column
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)

missing_df = pd.DataFrame({
 'Missing Count': missing,
 'Missing %': missing_pct
}).query('`Missing Count` > 0')

print("=== COLUMNS WITH MISSING VALUES ===")
print(missing_df)
print(f"\nTotal missing cells: {df.isnull().sum().sum()}")
print(f"Total cells : {df.size}")


In [ ]:
# Visualise missing value heatmap (text-based)
print("=== MISSING VALUE SUMMARY (per column) ===")
for col in df.columns:
 count = df[col].isnull().sum()
 if count > 0:
 bar = '' * count + '' * (10 - min(count, 10))
 print(f" {col:<20} {bar} {count} missing")


### 3b. Fill / Drop missing values

**Strategy:**
| Column | Strategy | Reason |
|--------|----------|--------|
| `Sales` | Fill with **median** | Numeric, outlier-robust |
| `Profit` | Fill with **median** | Numeric, outlier-robust |
| `Ship Mode` | Fill with **'Standard Class'** | Most common value (mode) |
| `Region` | Fill with **'Unknown'** | Categorical placeholder |


In [ ]:
# Create a clean copy — never modify the original
df_clean = df.copy()

# Fill numeric columns with median
df_clean = df_clean.assign(
 Sales=df_clean['Sales'].fillna(df_clean['Sales'].median()),
 Profit=df_clean['Profit'].fillna(df_clean['Profit'].median()),
)

# Fill categorical columns
df_clean['Ship Mode'] = df_clean['Ship Mode'].fillna('Standard Class')
df_clean['Region'] = df_clean['Region'].fillna('Unknown')

# Verify — should be 0
remaining_missing = df_clean.isnull().sum().sum()
print(f" Missing values handled.")
print(f" Missing before : {df.isnull().sum().sum()}")
print(f" Missing after : {remaining_missing}")


## Step 4 — Basic Operations: Filter Rows & Select Columns

### 4a. Filter rows — apply conditions


In [ ]:
# Filter 1: Orders with Sales > 500
high_sales = df_clean[df_clean['Sales'] > 500]
print(f"Orders with Sales > ₹500 : {len(high_sales)} rows")

# Filter 2: Technology category only
tech_orders = df_clean[df_clean['Category'] == 'Technology']
print(f"Technology category orders : {len(tech_orders)} rows")

# Filter 3: East region + Quantity >= 5
east_bulk = df_clean[(df_clean['Region'] == 'East') & (df_clean['Quantity'] >= 5)]
print(f"East region, Qty ≥ 5 : {len(east_bulk)} rows")

# Filter 4: Profitable orders (Profit > 0)
profitable = df_clean[df_clean['Profit'] > 0]
print(f"Profitable orders (P > 0) : {len(profitable)} rows")

# Show sample
print("\nSample filtered rows (Sales > 500):")
high_sales[['Order ID', 'Category', 'Region', 'Sales', 'Quantity', 'Profit']].head(5)


In [ ]:
# 4b. Select specific columns
selected = df_clean[['Order ID', 'Customer Name', 'Category', 'Sub-Category',
 'Region', 'Sales', 'Quantity', 'Discount', 'Profit']]

print(f"Selected {len(selected.columns)} columns from {len(df_clean.columns)} total")
print("\nSelected DataFrame (first 5 rows):")
selected.head()


## Step 5 — Remove Duplicates

Identify and remove exact duplicate rows.


In [ ]:
# Count duplicates before removal
dups_before = df_clean.duplicated().sum()
print(f"Duplicate rows found: {dups_before}")

# Show duplicate rows
if dups_before > 0:
 print("\nDuplicate rows:")
 print(df_clean[df_clean.duplicated(keep=False)][
 ['Order ID', 'Customer Name', 'Category', 'Sales', 'Quantity']
 ].head(10))

# Remove duplicates
df_clean = df_clean.drop_duplicates().reset_index(drop=True)

dups_after = df_clean.duplicated().sum()
print(f"\n Duplicates removed.")
print(f" Rows before : {len(df_clean) + dups_before}")
print(f" Rows after : {len(df_clean)}")
print(f" Removed : {dups_before} rows")


## Step 6 — Create a Derived Column: `total_amount`

We create `total_amount = Sales × Quantity` — the gross revenue for each order line.


In [ ]:
# Create derived column
df_clean['total_amount'] = (df_clean['Sales'] * df_clean['Quantity']).round(2)

print(" Derived column 'total_amount' created.")
print(f" Formula : total_amount = Sales × Quantity")
print(f"\n Min total_amount : {df_clean['total_amount'].min():>10.2f}")
print(f" Mean total_amount : {df_clean['total_amount'].mean():>10.2f}")
print(f" Max total_amount : {df_clean['total_amount'].max():>10.2f}")
print(f" Sum total_amount : {df_clean['total_amount'].sum():>10.2f}")

# Verify
print("\nSample rows showing new column:")
df_clean[['Order ID', 'Category', 'Sales', 'Quantity', 'total_amount']].head(8)


In [ ]:
# Distribution by category
print("=== Total Amount by Category ===")
cat_summary = df_clean.groupby('Category')['total_amount'].agg(['sum','mean','count'])
cat_summary.columns = ['Total Revenue', 'Avg per Order', 'Order Count']
cat_summary = cat_summary.sort_values('Total Revenue', ascending=False).round(2)
print(cat_summary)


## Step 7 — Save the Cleaned Dataset as a New CSV File


In [ ]:
# Save cleaned dataframe
output_path = 'superstore_cleaned.csv'
df_clean.to_csv(output_path, index=False)

print(f" Cleaned dataset saved to: {output_path}")
print(f" Rows : {df_clean.shape[0]}")
print(f" Columns : {df_clean.shape[1]}")
print(f" Columns : {list(df_clean.columns)}")


In [ ]:
# Quick validation — reload and verify
df_verify = pd.read_csv(output_path)
print("=== VERIFICATION (reloaded cleaned CSV) ===")
print(f"Shape : {df_verify.shape}")
print(f"Missing values : {df_verify.isnull().sum().sum()}")
print(f"Duplicates : {df_verify.duplicated().sum()}")
print(f"total_amount col: {'total_amount' in df_verify.columns}")
print("\nFirst 3 rows:")
df_verify.head(3)


## Summary — Data Cleaning Report

| Step | Task | Result |
|------|------|--------|
| 1 | Load CSV | 205 rows × 17 columns loaded |
| 2 | Exploration | head/tail/shape/dtypes/describe/info examined |
| 3 | Missing values | 32 nulls filled (median / mode / 'Unknown') |
| 4 | Filter & Select | Filtered by Sales, Category, Region; selected 9 key columns |
| 5 | Duplicates | 5 duplicate rows removed |
| 6 | Derived column | `total_amount = Sales × Quantity` created |
| 7 | Save CSV | `superstore_cleaned.csv` saved (200 rows × 18 columns) |

---


---
# Delta Lake MERGE Implementation

**Reference:** [Upsert into Delta Lake using merge — Microsoft Learn](https://learn.microsoft.com/en-us/azure/databricks/delta/merge)

Delta Lake MERGE (also called **upsert**) lets you atomically:
- **UPDATE** existing rows when a match is found
- **INSERT** new rows when no match exists 
- **DELETE** rows that no longer exist in the source (optional)

This is the standard pattern for **Change Data Capture (CDC)** and **SCD Type 1** pipelines.

## MERGE Syntax (SQL)

```sql
MERGE INTO target_table AS target
USING source_table AS source
ON target.key_column = source.key_column

WHEN MATCHED THEN
 UPDATE SET
 target.col1 = source.col1,
 target.col2 = source.col2

WHEN NOT MATCHED THEN
 INSERT (col1, col2, ...)
 VALUES (source.col1, source.col2, ...)

WHEN NOT MATCHED BY SOURCE THEN -- Databricks Runtime 12.2+
 DELETE
```


## Delta Lake MERGE — PySpark Implementation

The code below simulates a full MERGE pipeline using the **Superstore dataset**:
1. Create a Delta target table from the cleaned Superstore data
2. Simulate incoming updates (price changes, new orders)
3. Apply MERGE — update existing orders, insert new ones
4. Verify the results and show transaction log


In [ ]:
# 
# DELTA LAKE MERGE IMPLEMENTATION
# Reference: https://learn.microsoft.com/en-us/azure/databricks/delta/merge
#
# NOTE: This code runs on Databricks or a local PySpark + delta-spark setup.
# To run locally: pip install delta-spark pyspark
# To run on Databricks: paste directly into a notebook cell.
# 

# Uncomment the block below if running locally 
# from pyspark.sql import SparkSession
# from delta import configure_spark_with_delta_pip
# builder = (
# SparkSession.builder
# .appName("SuperstoreDeltaMerge")
# .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
# .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
# )
# spark = configure_spark_with_delta_pip(builder).getOrCreate()
# spark.sparkContext.setLogLevel("ERROR")

# On Databricks, 'spark' is already available 
# from delta.tables import DeltaTable
# from pyspark.sql import functions as F

print(" Spark session ready (Databricks environment)")
print(" delta-spark MERGE implementation follows below.")


In [ ]:
# 
# STEP 1: Create the TARGET Delta table
# Load cleaned Superstore CSV → write as Delta format
# 

DELTA_TARGET_PATH = "/tmp/delta/superstore_target"

# Load the cleaned CSV
target_df = spark.read.option("header", True).option("inferSchema", True) \
 .csv("superstore_cleaned.csv")

# Write as Delta table (initial load)
(target_df
 .write
 .format("delta")
 .mode("overwrite")
 .save(DELTA_TARGET_PATH))

# Verify
target_count = spark.read.format("delta").load(DELTA_TARGET_PATH).count()
print(f" Delta target table created at: {DELTA_TARGET_PATH}")
print(f" Row count: {target_count}")


In [ ]:
# 
# STEP 2: Simulate incoming source data (updates + new records)
# This represents a daily CDC feed with:
# - Updated Sales amounts for existing Order IDs
# - Brand new orders not yet in the target
# 

from pyspark.sql import functions as F

# 2a. Load existing orders to simulate updates (modify Sales for 10 orders)
existing_orders = spark.read.format("delta").load(DELTA_TARGET_PATH).limit(10)

updated_orders = existing_orders.withColumn(
 "Sales", F.round(F.col("Sales") * 1.15, 2) # 15% price increase
).withColumn(
 "Profit", F.round(F.col("Profit") * 1.10, 2) # 10% profit increase
).withColumn(
 "total_amount", F.round(F.col("Sales") * F.col("Quantity"), 2)
)

# 2b. Create 5 brand-new orders (not in target)
new_orders_data = [
 ("CA-2024-999001", "2024-01-15", "2024-01-18", "First Class",
 "CG-99001", "New Customer A", "Consumer", "East",
 "TEC-8801", "Technology", "Phones", "iPhone Pro 2024",
 1299.99, 2, 0.0, 520.0),
 ("CA-2024-999002", "2024-01-16", "2024-01-20", "Standard Class",
 "CG-99002", "New Customer B", "Corporate", "West",
 "FUR-8802", "Furniture", "Chairs", "Ergonomic Chair X",
 449.99, 4, 0.1, 90.0),
 ("CA-2024-999003", "2024-01-17", "2024-01-21", "Second Class",
 "CG-99003", "New Customer C", "Home Office", "Central",
 "OFF-8803", "Office Supplies", "Binders", "Heavy Duty Binder",
 24.99, 10, 0.0, 8.5),
 ("CA-2024-999004", "2024-01-18", "2024-01-22", "Same Day",
 "CG-99004", "New Customer D", "Consumer", "South",
 "TEC-8804", "Technology", "Accessories", "USB-C Hub 7-in-1",
 79.99, 6, 0.2, 15.0),
 ("CA-2024-999005", "2024-01-19", "2024-01-23", "Standard Class",
 "CG-99005", "New Customer E", "Corporate", "East",
 "FUR-8805", "Furniture", "Tables", "Standing Desk Pro",
 899.0, 1, 0.0, 250.0),
]

cols = ["Order ID","Order Date","Ship Date","Ship Mode","Customer ID",
 "Customer Name","Segment","Region","Product ID","Category",
 "Sub-Category","Product Name","Sales","Quantity","Discount",
 "Profit"]

new_orders_df = spark.createDataFrame(new_orders_data, cols) \
 .withColumn("Row ID", F.lit(-1)) \
 .withColumn("total_amount", F.round(F.col("Sales") * F.col("Quantity"), 2))

# 2c. Union updates + new orders → this is the SOURCE DataFrame
source_df = updated_orders.unionByName(new_orders_df, allowMissingColumns=True)

print(f" Source DataFrame prepared:")
print(f" Updated orders : 10")
print(f" New orders : 5")
print(f" Total source : {source_df.count()} rows")


In [ ]:
# 
# STEP 3: Execute MERGE (Upsert)
# - WHEN MATCHED → UPDATE the row (Sales, Profit, total_amount)
# - WHEN NOT MATCHED → INSERT the new row
#
# Reference: https://learn.microsoft.com/en-us/azure/databricks/delta/merge
# 

from delta.tables import DeltaTable

# Load the target Delta table
deltaTable = DeltaTable.forPath(spark, DELTA_TARGET_PATH)

# Execute MERGE
(deltaTable.alias("target")
 .merge(
 source_df.alias("source"),
 "target.`Order ID` = source.`Order ID`" # join key
 )
 .whenMatchedUpdate(set={
 "Sales" : "source.Sales",
 "Profit" : "source.Profit",
 "total_amount" : "source.total_amount",
 "Discount" : "source.Discount",
 })
 .whenNotMatchedInsertAll() # insert all columns for new rows
 .execute()
)

print(" MERGE executed successfully!")
print()

# Verify final counts
final_df = spark.read.format("delta").load(DELTA_TARGET_PATH)
final_count = final_df.count()
print(f" Rows before MERGE : {target_count}")
print(f" New rows inserted : 5")
print(f" Rows updated : 10")
print(f" Rows after MERGE : {final_count}")


In [ ]:
# 
# STEP 4: Verify MERGE results
# Show updated rows and newly inserted rows
# 

from pyspark.sql import functions as F

merged_df = spark.read.format("delta").load(DELTA_TARGET_PATH)

# Show new rows (our 5 new orders)
print("=== NEWLY INSERTED ROWS (Order IDs starting with CA-2024-999) ===")
new_rows = merged_df.filter(F.col("Order ID").startswith("CA-2024-999"))
new_rows.select("Order ID", "Customer Name", "Category", "Sales",
 "Quantity", "total_amount", "Region").show(truncate=False)

# Show sample updated rows (Sales increased by 15%)
print("=== SAMPLE UPDATED ROWS (first 5 of the 10 updated) ===")
updated_sample = merged_df.limit(5)
updated_sample.select("Order ID", "Sales", "Profit", "total_amount").show()


In [ ]:
# 
# STEP 5: View Delta Lake Transaction Log (History)
# Delta tracks every operation — SELECT, INSERT, UPDATE, MERGE, etc.
# 

print("=== DELTA LAKE TRANSACTION HISTORY ===")
deltaTable.history().select(
 "version", "timestamp", "operation", "operationMetrics"
).show(truncate=False)


In [ ]:
# 
# STEP 6: Time Travel — read the table AS OF a previous version
# Delta Lake lets you query any previous snapshot
# 

# Read version 0 (before MERGE)
df_v0 = spark.read.format("delta") \
 .option("versionAsOf", 0) \
 .load(DELTA_TARGET_PATH)

# Read version 1 (after MERGE)
df_v1 = spark.read.format("delta") \
 .option("versionAsOf", 1) \
 .load(DELTA_TARGET_PATH)

print("=== TIME TRAVEL COMPARISON ===")
print(f"Version 0 (before MERGE) : {df_v0.count()} rows")
print(f"Version 1 (after MERGE) : {df_v1.count()} rows")
print(f"Difference : {df_v1.count() - df_v0.count()} new rows added")


In [ ]:
# 
# STEP 7: SQL MERGE — equivalent SQL syntax on Databricks
# 

# Register DataFrames as temporary views
spark.read.format("delta").load(DELTA_TARGET_PATH).createOrReplaceTempView("superstore_target")
source_df.createOrReplaceTempView("superstore_source")

sql_merge = (
 "MERGE INTO superstore_target AS target "
 "USING superstore_source AS source "
 "ON target.`Order ID` = source.`Order ID` "
 "WHEN MATCHED THEN UPDATE SET "
 "target.Sales = source.Sales, target.Profit = source.Profit, target.total_amount = source.total_amount "
 "WHEN NOT MATCHED THEN INSERT *"
)
spark.sql(sql_merge)

print(" SQL MERGE executed successfully")
print(" This is the SQL-equivalent of the PySpark DeltaTable.merge() above")


## Delta Lake MERGE — Summary

| Clause | When it runs | What it does |
|--------|-------------|--------------|
| `WHEN MATCHED` | Source row matches target on key | UPDATE specified columns |
| `WHEN NOT MATCHED` | Source row has no match in target | INSERT as a new row |
| `WHEN NOT MATCHED BY SOURCE` | Target row has no match in source | DELETE or UPDATE (Databricks 12.2+) |

### Key Benefits of Delta Lake MERGE

- **ACID transactions** — all-or-nothing, no partial writes
- **Optimistic concurrency** — concurrent reads never blocked
- **Time travel** — query any prior version with `versionAsOf`
- **Schema evolution** — auto-merge schema changes with `spark.databricks.delta.schema.autoMerge.enabled = true`
- **Deduplication** — use `WHEN NOT MATCHED` with insert-only merge to avoid duplicate records

### Common Use Cases

| Use Case | MERGE Pattern |
|----------|--------------|
| SCD Type 1 (overwrite) | MATCHED → UPDATE, NOT MATCHED → INSERT |
| SCD Type 2 (history) | MATCHED → INSERT new + close old row |
| CDC (Change Data Capture) | MATCHED → UPDATE/DELETE, NOT MATCHED → INSERT |
| Deduplication | NOT MATCHED → INSERT only |
| Incremental sync | MATCHED BY SOURCE → DELETE, others normal |

---

**References:**
- [Upsert into Delta Lake — Microsoft Learn](https://learn.microsoft.com/en-us/azure/databricks/delta/merge)
- [Delta Lake OSS Documentation](https://docs.delta.io/delta-update/)
- [Superstore Dataset — Kaggle](https://www.kaggle.com/datasets/vivek468/superstore-dataset-final)
